# UML-Pipeline — free GPU scoring on Google Colab

This notebook scores **already rendered** UML PNGs with **Qwen2.5-VL-3B** on a free Colab T4.
It uses the same 0–6 `SCORE:` / `EXPLANATION:` protocol and MMMU weight **53.1** as the thesis app.

**What Colab free can run**
- Qwen2.5-VL-3B (paper VLM #1) — fits a T4

**What Colab free cannot run well**
- LLaMA-3.2-11B-Vision (~paper VLM #2)
- Aya-Vision-8B (~paper VLM #3, ~17GB)
Those need Colab Pro / A100, a campus GPU, or a paid VM.

**Steps**
1. Runtime → Change runtime type → **T4 GPU**
2. On your Mac: `python scripts/export_colab_pack.py --limit 20`
3. Upload `data/colab_pack.zip` in the next cells
4. Run all cells → download `colab_vlm_scores.csv`

In [ ]:
import torch, shutil, os
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('NO GPU — Runtime → Change runtime type → T4 GPU, then rerun.')

In [ ]:
%pip install -q transformers==4.51.3 accelerate qwen-vl-utils pillow pandas
print('deps ready')

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, shutil

WORK = Path('/content/uml_pack')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

print('Upload data/colab_pack.zip from your Mac (scripts/export_colab_pack.py)')
uploaded = files.upload()
zpath = next(iter(uploaded))
with zipfile.ZipFile(zpath) as zf:
    zf.extractall(WORK)
manifest = json.loads((WORK / 'manifest.json').read_text())
print('diagrams:', len(manifest))
print(manifest[0].keys())

In [ ]:
import re
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

WEIGHT_QWEN = 53.1
TAU = 4
PROMPT = '''You are evaluating a UML diagram image against a technical specification.

Specification:
{specification}

Score the diagram from 0 to 6 using these criteria jointly:
1. Semantic correctness
2. Structural completeness
3. Syntactic accuracy
4. Overall coherence

Respond in exactly this format:
SCORE: <integer 0-6>
EXPLANATION: <2-4 sentences>'''

def parse_score(text: str):
    m = re.search(r'(?im)^\s*SCORE\s*[:\-]\s*([0-6])\b', text or '')
    score = int(m.group(1)) if m else 0
    e = re.search(r'(?is)^\s*EXPLANATION\s*[:\-]\s*(.+)', text or '')
    expl = ' '.join(e.group(1).split())[:800] if e else (text or '')[:400]
    return score, expl

model_id = 'Qwen/Qwen2.5-VL-3B-Instruct'
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
    trust_remote_code=True,
)
print('loaded', model_id)

In [ ]:
rows = []
for i, item in enumerate(manifest, 1):
    img_path = WORK / item['image']
    spec = (item.get('specification') or item.get('requirement') or '')[:3500]
    user_text = PROMPT.format(specification=spec)
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': str(img_path)},
            {'type': 'text', 'text': user_text},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt')
    inputs = inputs.to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=220)
    trimmed = out[:, inputs.input_ids.shape[1]:]
    raw = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
    score, expl = parse_score(raw)
    rows.append({
        'artifact_id': item.get('artifact_id'),
        'diagram_type': item.get('diagram_type'),
        'qwen25vl3b': score,
        'weight': WEIGHT_QWEN,
        'vote_tau4': int(score >= TAU),
        'explanation': expl,
        'raw': raw[:500],
    })
    print(f"[{i}/{len(manifest)}] #{item.get('artifact_id')} {item.get('diagram_type')} SCORE={score}")

import pandas as pd
df = pd.DataFrame(rows)
df['S_qwen_only'] = df['qwen25vl3b']  # full paper S needs 3 VLMs; this is VLM #1 only
display(df[['artifact_id','diagram_type','qwen25vl3b','vote_tau4']])
csv_path = '/content/colab_vlm_scores.csv'
df.to_csv(csv_path, index=False)
print('saved', csv_path)
files.download(csv_path)

## Limits (do not hide these)

- This is **one** of the three paper VLMs (Qwen2.5-VL-3B, weight 53.1).
- Dataset gate `A=1` in the thesis needs **≥2** VLMs at score ≥ 4. A Colab-only Qwen score cannot replace the full ensemble.
- Keep interactive Generate on the Mac (skip VLMs). Use this notebook for **paper tables / batch scoring** of PNGs you already rendered.
- If Colab disconnects, re-upload the zip and rerun from the model-load cell.